In [120]:
# Imports

from agents import Agent, Runner, function_tool, trace
from openai.types.responses import ResponseTextDeltaEvent
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv
from agents import WebSearchTool
from agents import trace
from IPython.display import display, Markdown
import os
import asyncio

In [121]:
#Load environmental Variables
load_dotenv(override=True)


True

In [122]:
LIGHT_MODEL = "gpt-4.1-mini"
STRONG_MODEL = "gpt-4.1"

In [123]:
# Classification Model
class Clarifying_Questions(BaseModel):
    questions: List[str]

#Search Plan Model
class SearchPlan(BaseModel):
    queries: List[str]

#Evaluation Model
class Evaluation(BaseModel):
    is_sufficient: bool
    relevance_score: float  # 0–1
    coverage_score: float
    depth_score: float
    feedback: str
    refined_queries: List[str]

#Research Output Model
class ResearchOutput(BaseModel):
    summary: str
    key_points: List[str]

class ReportOutput(BaseModel):
    report: str

class ManagerDecision(BaseModel):
    action: str
    reasoning: str



In [124]:
#Classification Agent

CLASSIFICATION_INSTRUCTIONS = """
You are a research assistant.

Given a user query, generate exactly 3 concise clarifying questions
that would help improve the quality and relevance of a deep research task.

Rules:
- Ask exactly 3 questions
- Questions must be specific and useful for refining search
- Do not answer the query
- Do not add explanations
"""

clarification_agent = Agent(
    name="Clarification Agent",
    instructions= CLASSIFICATION_INSTRUCTIONS,
    model=LIGHT_MODEL,
    output_type=Clarifying_Questions
)

In [125]:
#Testing

result = await Runner.run(clarification_agent, "Impact of AI on jobs")
print(result.final_output)

questions=['Are you interested in the impact of AI on jobs globally or in a specific region?', 'Do you want to focus on a particular industry or sector affected by AI in terms of employment?', 'Are you looking for current impacts, future projections, or historical trends of AI on jobs?']


In [126]:
#Classification Tool
async def generate_clarifying_questions(query: str) -> Clarifying_Questions:
    """
    Generate exactly 3 clarifying questions to improve a research query.
    """
    result = await Runner.run(clarification_agent, query)
    return result.final_output


clarification_tool = function_tool(generate_clarifying_questions)

In [127]:
## Stratergic REasoning Agents

STRATERGIC_INSTRUCTIONS = """
You are an expert research planner.

Given a user query and clarifying questions, generate a list of effective search queries.

Rules:
- Generate 3 to 5 search queries
- Queries should be specific and diverse
- Incorporate clarifying questions into the queries
- Focus on maximizing useful information retrieval
- Do not explain anything
"""

search_planning_agent = Agent(
    name="Search Planning Agent",
    instructions=STRATERGIC_INSTRUCTIONS,
    model=LIGHT_MODEL,
    output_type=SearchPlan
)

In [128]:
#Testing
result = await Runner.run(search_planning_agent, "Query: AI jobs")
print(result.final_output)
print(result.final_output.queries)

queries=['types of AI jobs and required skills', 'career progression in artificial intelligence field', 'companies hiring for AI roles in 2024', 'entry-level AI job opportunities', 'impact of AI automation on job market']
['types of AI jobs and required skills', 'career progression in artificial intelligence field', 'companies hiring for AI roles in 2024', 'entry-level AI job opportunities', 'impact of AI automation on job market']


In [129]:
#Search Planning Agent Tool
async def generate_search_queries(input_text: str) -> SearchPlan:
    """
    Generate 3–5 optimized search queries based on a user query and clarifications.
    """
    result = await Runner.run(search_planning_agent, input_text)
    return result.final_output

search_planning_tool = function_tool(generate_search_queries)

In [130]:
#Intanciate the Web Search Tool
web_search_tool = WebSearchTool()

In [131]:
RESEARCH_INSTRUCTIONS = """
You are a deep research analyst.

Your job is to analyze search results and produce a high-quality synthesis.

Rules:
- Provide a clear and concise summary
- Extract key insights as bullet points
- Focus on relevance to the original query
- Avoid repetition
- Do not include irrelevant information
"""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_INSTRUCTIONS,
    model=STRONG_MODEL,
    output_type=ResearchOutput
)

In [132]:
##Evaluator Agent
EVALUATION_INSTRUCTIONS = """
You are a strict research evaluator.

Evaluate the research output using the following criteria:

1. Relevance (0 to 1)
2. Coverage (0 to 1)
3. Depth (0 to 1)

Rules:
- You MUST return scores as floats between 0 and 1
- Do NOT return 0 unless the output is completely irrelevant
- Be realistic in scoring (most good outputs should be 0.6–0.9 range)

Also:
- Provide feedback explaining weaknesses
- Suggest improved search queries if needed

IMPORTANT:
Return valid structured output matching the schema exactly.
"""
evaluator_agent = Agent(
    name="Evaluator Agent",
    instructions=EVALUATION_INSTRUCTIONS,
    model=STRONG_MODEL,
    output_type=Evaluation
)

In [133]:
SEARCH_INSTRUCTIONS = """You perform web searches.

Given a query:
- Use the web search tool
- Return relevant results clearly
"""

search_agent = Agent(
    name="Search Agent",
    model=LIGHT_MODEL,
    tools=[web_search_tool],
    instructions=SEARCH_INSTRUCTIONS
)

In [134]:
#Manager Agent

MANAGER_INSTRUCTIONS = """
You are an autonomous research manager.

Your job is to decide the next best action in a research workflow.

Available actions:
- clarify → ask clarifying questions
- plan → generate search queries
- search → perform web search
- analyze → synthesize results
- evaluate → evaluate quality
- refine → improve query
- stop → finish process

Rules:
- Think step-by-step
- Choose the best next action
- Be efficient but thorough
- Stop when result quality is high

Return only the next action and reasoning.
"""
manager_agent = Agent(
    name="Deep Research Manager",
    model=STRONG_MODEL,
    instructions=MANAGER_INSTRUCTIONS,
    handoffs=[
        clarification_agent,
        search_planning_agent,
        search_agent,          # ✅ ADD THIS
        research_agent,
        evaluator_agent
    ],
    output_type=ManagerDecision
)

In [135]:
REPORT_INSTRUCTIONS = """
You are a professional research writer.

Your job is to transform research insights into a comprehensive, well-structured report.

Requirements:
- Write a detailed report (~800–1200 words)
- Include:
    1. Introduction
    2. Main Body (with clear sections and headings)
    3. Conclusion
- Expand on ideas with explanations, examples, and context
- Ensure clarity, depth, and logical flow
- Avoid bullet points unless necessary
- Write in a professional and engaging tone
"""

report_agent = Agent(
    name="Report Agent",
    model=STRONG_MODEL,
    instructions=REPORT_INSTRUCTIONS,
    output_type=ReportOutput
)

In [136]:
#EVALUATION_THRESHOLD
RELEVANCE_THRESHOLD = 0.75
COVERAGE_THRESHOLD = 0.7
DEPTH_THRESHOLD = 0.7
HIGH_CONFIDENCE = 0.9
MAX_ITERATIONS = 3

In [137]:
#Get Classifications
async def get_clarifications(query: str):
    result = await generate_clarifying_questions(query)
    questions = result.questions
    print("Clarifications:", questions)
    return questions

In [138]:
#Search Planning
async def plan_searches(query: str, clarifications: list[str]):
    enriched_input = f"""
Query: {query}

Clarifications:
{chr(10).join(clarifications)}
"""
    result = await generate_search_queries(enriched_input)
    queries = result.queries
    print("Search Queries:", queries)
    return queries

In [139]:
#Execute Searches
async def execute_searches(queries: list[str]):
    print("Running parallel web searches...")

    async def run_single_query(q):
        try:
            print(f"Searching: {q}")
            result = await Runner.run(search_agent, q)
            return str(result.final_output)
        except Exception as e:
            print(f"Error searching '{q}': {e}")
            return ""

    tasks = [run_single_query(q) for q in queries]
    results = await asyncio.gather(*tasks)

    combined_results = "\n\n".join(r for r in results if r)
    print("Combined search results collected.")

    return combined_results

In [140]:
#Research Analysis
async def analyze_results(combined_results: str):
    result = await Runner.run(research_agent, combined_results)
    research_output = result.final_output
    print("Research Summary:", research_output.summary)
    print("Research Key Points:", research_output.key_points)
    return research_output

In [141]:
#Output Evaluation
async def evaluate_output(original_query: str, research_output):
    evaluation_input = f"""
Original Query: {original_query}

Research Output:
Summary: {research_output.summary}
Key Points: {research_output.key_points}
"""
    result = await Runner.run(evaluator_agent, evaluation_input)
    evaluation = result.final_output
    print("Evaluation:", evaluation)
    return evaluation


In [142]:
#Score Calculation
def compute_score(evaluation):
    score = (
        0.4 * evaluation.relevance_score +
        0.3 * evaluation.coverage_score +
        0.3 * evaluation.depth_score
    )
    print("Weighted Score:", score)
    return score

In [143]:
#Stop Condition
def should_stop(evaluation, score):
    if score >= HIGH_CONFIDENCE:
        print("High confidence result reached.")
        return True

    if (
        evaluation.relevance_score >= RELEVANCE_THRESHOLD and
        evaluation.coverage_score >= COVERAGE_THRESHOLD and
        evaluation.depth_score >= DEPTH_THRESHOLD
    ):
        print("Thresholds met.")
        return True

    return False

In [144]:
#Refine Query
def refine_query(current_query: str, evaluation):
    if evaluation.refined_queries:
        refined_query = " ".join(evaluation.refined_queries)
        print("Refined Query:", refined_query)
        return refined_query

    fallback_query = current_query + " more detailed analysis"
    print("Fallback Refined Query:", fallback_query)
    return fallback_query

In [ ]:
async def deep_research(query: str):
    current_query = query

    best_output = None
    best_score = 0

    # 🧠 STATE TRACKING
    clarifications = None
    search_queries = None
    combined_results = None
    research_output = None
    evaluation = None

    for step in range(MAX_ITERATIONS):
        print(f"\n=== Iteration {step + 1} ===")

        # 🧠 Manager decides next step
        decision_input = f"""
Current Query: {current_query}

State:
Clarifications: {clarifications}
Search Queries: {search_queries}
Has Results: {combined_results is not None}
Has Analysis: {research_output is not None}
Has Evaluation: {evaluation is not None}
"""

        decision_result = await Runner.run(manager_agent, decision_input)
        decision = decision_result.final_output

        print("Manager Decision:", decision)

        # 🔁 ACTION ROUTING

        if decision.action == "clarify":
            clarifications = await get_clarifications(current_query)

        elif decision.action == "plan":
            clarifications = clarifications or []
            search_queries = await plan_searches(current_query, clarifications)

        elif decision.action == "search":
            if search_queries:
                combined_results = await execute_searches(search_queries)

        elif decision.action == "analyze":
            if combined_results:
                research_output = await analyze_results(combined_results)

        elif decision.action == "evaluate":
            if research_output:
                evaluation = await evaluate_output(query, research_output)

                # DEBUG
                print("Raw Evaluation Object:", evaluation)
                print("Scores:",
                      evaluation.relevance_score,
                      evaluation.coverage_score,
                      evaluation.depth_score)

                score = compute_score(evaluation)

                if score > best_score:
                    best_score = score
                    best_output = research_output

                if should_stop(evaluation, score):
                    print("✅ Manager decided to stop")
                    return best_output

        elif decision.action == "refine":
            if evaluation:
                current_query = refine_query(current_query, evaluation)

        elif decision.action == "stop":
            print("🛑 Manager forced stop")
            break

        else:
            print("⚠️ Unknown action — continuing")

    # ✅ fallback safety
    if best_output is None:
        raise ValueError("No valid research output generated.")

    print("⚠️ Max iterations reached — returning best result")
    return best_output

In [146]:
#Runner Function

async def run_and_display(query: str):
    result = await deep_research(query)

    if result is None:
        print("❌ No result returned from deep_research")
        return

    markdown_output = f"""
# 🔍 Deep Research Result

## 📌 Query
{query}

## 🧠 Summary
{result.summary}

## 🔑 Key Insights
{"".join([f"- {point}\n" for point in result.key_points])}
"""

    display(Markdown(markdown_output))

In [147]:
async def generate_report(query: str, research_output):
    input_text = f"""
Query: {query}

Summary:
{research_output.summary}

Key Points:
{chr(10).join(research_output.key_points)}
"""

    result = await Runner.run(report_agent, input_text)
    return result.final_output.report

In [148]:
query = "Latest AI Agent frameworks in 2025"

await run_and_display(query)


--- Iteration 1 ---
Clarifications: ['Are you interested in AI agent frameworks for a specific application area, such as robotics, virtual assistants, or gaming?', 'Do you want information on open-source frameworks or commercial/proprietary AI agent frameworks?', 'Are you looking for frameworks that focus on particular AI techniques, such as reinforcement learning, natural language processing, or multi-agent systems?']
Search Queries: ['Latest AI agent frameworks for robotics in 2025', 'Top open-source AI agent frameworks for virtual assistants in 2025', 'Commercial AI agent frameworks focusing on reinforcement learning in 2025', 'AI agent frameworks specialized in multi-agent systems for gaming in 2025', '2025 advancements in natural language processing AI agent frameworks']
Running parallel web searches...
Searching: Latest AI agent frameworks for robotics in 2025
Searching: Top open-source AI agent frameworks for virtual assistants in 2025
Searching: Commercial AI agent frameworks 


# 🔍 Deep Research Result

## 📌 Query
Latest AI Agent frameworks in 2025

## 🧠 Summary
As of March 2026, the landscape of AI agent frameworks has rapidly evolved, delivering advanced solutions across robotics, virtual assistants, reinforcement learning systems, gaming, and natural language processing. Industry collaborations (notably Nvidia's Nemotron Coalition), open-source initiatives, standardized interoperability protocols (like Agent2Agent), and a focus on modularity and dynamic adaptation have defined this era. There is a clear trend toward scalable, accessible, and explainable frameworks, modular skill libraries, and robust multi-agent collaboration, which enable AI agents to operate effectively in real-world and virtual environments.

## 🔑 Key Insights
- Nvidia's Nemotron Coalition unites major AI labs to develop open frontier models—Nemotron 4 and Isaac GR00T N1.7—focused on enhancing reasoning and VLA capabilities in humanoid robotics.
- RoboOS introduces a hierarchical Brain–Cerebellum architecture, offering robust multi-agent planning, modular skill execution, and real-time shared memory for coordination.
- The RAI framework facilitates embodied multi-agent systems, providing native integrations with robotics stacks, LLMs, and simulators, and is proven on both physical hardware and simulations.
- Polymorphic Combinatorial Framework (PCF) uses mathematically grounded adaptive agents, enabling real-time reconfiguration and explainable, ethical operations across domains like robotics and collaborative systems.
- Agent2Agent (A2A) protocol standardizes inter-agent communication, allowing agents from different vendors/frameworks to interoperate, discover, and coordinate tasks.
- Top open-source frameworks for virtual assistants include LangChain (integration powerhouse), Rasa (conversational AI specialist), Cognitive Kernel-Pro (modular agentic design), LightAgent (lightweight, memory-centric), and OpenClaw (autonomous workflow via messaging).
- Commercial RL agent frameworks—Agent Lightning, ToolBrain, Adaptive ML, OpenAI Agents SDK—offer flexible, modular training for large language agents and robust agent monitoring.
- Advances in gaming-focused multi-agent frameworks: OpenClaw (modular agent skill system), AutoGen (event-driven collaborative agents), LangGraph (workflow orchestration), CrewAI (collaborative NPCs), and Agent2Agent for cross-framework compatibility.
- Standardization efforts with the Model Context Protocol (MCP) and Agent2Agent ensure broad interoperability among AI and LLM-based agent systems, fostering collaborative AI environments.
- New paradigms like Model-First Reasoning (MFR) and frameworks such as PFEA and AutoAgent push the boundaries of natural language planning, feedback, and zero-code, user-friendly agent creation.

